In [ ]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(".env 파일에 GROQ_API_KEY 가 없습니다.")
print("GROQ_API_KEY 설정 확인 완료", GROQ_API_KEY[:2])


#### Groq API로 호출 가능한 모델 목록 확인

- Groq는 OpenAI 호환 API이므로 `openai` SDK에 `base_url`만 Groq 주소로 지정하면 `client.models.list()`로 모델 목록을 조회할 수 있습니다.
- 목록 조회는 토큰을 사용하지 않습니다.
- 아래 `ChatOpenAI`에서 사용할 모델명이 목록에 있는지 확인하는 용도로도 사용할 수 있습니다.

In [ ]:
# openai SDK 불러오기 (Groq도 OpenAI 호환 API라서 같은 SDK 사용)
from openai import OpenAI

GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트

# Groq 키와 엔드포인트로 클라이언트 생성 (GROQ_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

In [ ]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

In [ ]:
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-3.5-turbo-0125")

# Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    # model="qwen/qwen3.8-27b",
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

In [ ]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

In [ ]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [ ]:
print(type(response))
print(response)